# Setup · Your platform

Run this notebook once, top to bottom. It takes about 45 minutes, most of it waiting for Azure.

| Step | What happens | Time |
|---|---|---|
| 1 | Check your tools | 1 min |
| 2 | Sign in to Azure | 2 min |
| 3 | Tell Terraform who you are | 1 min |
| 4 | Create your resources with Terraform | 6 min |
| 5 | Connect the notebooks to your resources | 1 min |
| 6 | Call a model, with no API key | 1 min |
| 7 | Build and deploy the service | 5 min |
| 8 | Score it with the harness | 2 min |

Every helper in `course` prints the exact command it runs. You can copy any of them into a terminal opened in `fy26/`.

## 1 · Check your tools

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course.py").exists())))
import course

course.setup()
course.doctor()

Every tool should show a version. If one says `MISSING`, install it from the link, restart VS Code, and run the cell again.

## 2 · Sign in to Azure

`az login` opens a browser. Sign in with the account that owns your course subscription. If no browser opens, use `az login --use-device-code` instead.

In [ ]:
!az login --output none

In [ ]:
course.run(["az", "account", "show", "--query", "{subscription:name, id:id, user:user.name}", "-o", "table"])

If this is not the subscription you want, switch with `!az account set --subscription "<name or id>"` and run the cell above again. Terraform deploys into whatever subscription is shown here.

## 3 · Tell Terraform who you are

Terraform reads its inputs from `infra/terraform.tfvars`. Your `student_id` (use your NetID) appears in every resource name and is how we find your submission.

**Where your resources go.** If the class created a resource group for you (in the Northwestern class subscription it is named like `<netid>_resourcegroup`), put its name in `RESOURCE_GROUP`: you are Owner of that group, but you cannot create new groups. If you own the whole subscription, leave it empty and Terraform creates `rg-nu-<STUDENT_ID>`. The group's own region does not matter; your resources go to `LOCATION`.

**Your region is picked for you.** A subscription can hold at most 50 Container Apps environments per region, and the class is larger than that, so the class is split in two by the last digit of your NetID: 0 to 4 use `eastus2`, 5 to 9 use `eastus`. Both regions offer the same models and features. Do not change it unless an instructor asks you to; the class subscription allows US regions only.

In [ ]:
STUDENT_ID = "abc1234"  # your NetID: 3-12 lowercase letters or digits, starting with a letter
RESOURCE_GROUP = ""  # a group created for you, such as "<netid>_resourcegroup"; leave "" to let Terraform create one
LOCATION = "eastus2" if STUDENT_ID[-1] in "01234" else "eastus"  # the class is split across two regions

print(f"Region for {STUDENT_ID}: {LOCATION}")
course.write_tfvars(STUDENT_ID, LOCATION, resource_group=RESOURCE_GROUP, create_resource_group=not RESOURCE_GROUP)

## 4 · Create your resources with Terraform

Terraform works in three steps. Read the output of each one: that is how you learn the tool.

- **init** downloads the Azure provider (the plug-in that knows how to talk to Azure).
- **plan** compares the code in `infra/*.tf` with what exists in Azure and lists the changes. Nothing is created yet.
- **apply** makes exactly the changes in the plan.

Primer: [Foundations · Terraform](../foundations/terraform.ipynb).

In [ ]:
course.tf("init", "-input=false")

In [ ]:
course.tf("plan", "-input=false", "-out=tfplan")

The last line should read `Plan: 12 to add, 0 to change, 0 to destroy` (13 if Terraform also creates your resource group). Scroll up and find the Foundry account (`azurerm_cognitive_account.foundry`) and the two model deployments. Then apply the plan. It takes about 5 minutes.

If `plan` or `apply` stops with an error, find its code (`AuthorizationFailed`, `InsufficientQuota`, `MissingSubscriptionRegistration`…) in the troubleshooting table of the [Setup README](README.md#if-something-goes-wrong). Fix the cause, then run the plan cell and the apply cell again: Terraform keeps what it already created and continues from there.

In [ ]:
course.tf("apply", "-input=false", "tfplan")

## 5 · Connect the notebooks to your resources

Terraform outputs are the facts the rest of the course needs: endpoints, model names, registry name. This writes them to `fy26/.env`, which the service and every notebook read.

In [ ]:
course.write_env()

## 6 · Call a model, with no API key

`DefaultAzureCredential` finds your `az login` session and exchanges it for a short-lived token. Foundry accepts the token because Terraform gave you the **Foundry User** role on the resource. In Azure, the service does the same with its managed identity.

If you get a 401 or 403, the role assignment is still propagating: wait two minutes and run the cell again.

In [ ]:
import os

client = course.openai_client()
response = client.responses.create(
    model=os.environ["FOUNDRY_MODEL"],
    input="In one sentence: what is a bike-share system?",
)
print(response.output_text)
print(f"\nTokens: {response.usage.input_tokens} in, {response.usage.output_tokens} out")

## 7 · Build and deploy the service

`course.deploy()` does three things, printing each command:

1. `az acr build` uploads `app/`, `pyproject.toml` and `uv.lock`, and builds the image **in Azure** using the `Dockerfile`.
2. It writes the new image tag to `infra/image.auto.tfvars` and runs `terraform apply`, which creates or updates the Container App.
3. It waits until the new revision answers with the new version.

The first deployment takes about 5 minutes.

In [ ]:
url = course.deploy()
print(url)

Open the URL with `/docs` appended in your browser: FastAPI generates an interactive page for every route.

## 8 · Score it with the harness

The harness checks that your service is up, identifies itself, and handles the Week 5 tasks. The starting code is naive on purpose: expect a score in the 80s. In Week 5 you will raise it.

In [ ]:
proof = course.check(5, url)

## Done

You now have a working platform. From here on, every week ends with the same two cells: `course.deploy()` and `course.check(week)`.

| Resource | Name pattern | Purpose |
|---|---|---|
| Resource group | `rg-nu-<student_id>` | Container for everything below |
| Foundry account + project | `aif-…`, `proj-<student_id>` | Models and agents |
| Model deployments | `gpt-5.4-mini`, `text-embedding-3-small` | What your code calls by name |
| Container Registry | `acr…` | Your images |
| Container Apps environment + app | `cae-…`, `ca-…` | Runs the service at an HTTPS URL |
| Managed identity | `id-…` | Lets the app pull images and call Foundry |
| Log Analytics + Application Insights | `log-…`, `appi-…` | Logs, metrics, traces |

At the end of the course, remove everything with `course.tf("destroy")`.